# OPENBREWERY - SILVER NOTEBOOK

# 1. GENERAL SETTINGS

## 1.1 Load Variables

In [0]:
%run ./00-Config

## 1.2 Imports

In [0]:
from delta.tables import DeltaTable

from pyspark.sql import functions as F

from pyspark.sql.window import Window

from pyspark.sql.types import *

from datetime import datetime

## 1.3 Notebook Variables

In [0]:
bronze_table = (
    f"{catalog_name}.bronze.openbrewery_bronze"
)

silver_table = (
    f"{catalog_name}.silver.openbrewery_silver"
)

control_table_name = (
    f"{catalog_name}.control.openbrewery_control"
)

monitoring_table = (
    f"{catalog_name}.control.pipeline_monitoring"
)

data_quality_table_name = (
    data_quality_table
)

quarantine_table_name = (
    quarantine_silver_table
)

pipeline_name = "openbrewery_silver"

layer = "silver"

execution_id = datetime.utcnow().strftime(
    "%Y%m%d%H%M%S"
)

start_time = datetime.utcnow()

## 1.4 Get Last Timestamp Processed

In [0]:
control_df = spark.sql(f"""

SELECT
    MAX(last_ingestion_timestamp) AS last_ts

FROM {control_table_name}

WHERE pipeline_name = 'openbrewery_silver'

""")

last_ts = control_df.collect()[0]["last_ts"]

print("=" * 60)
print(f"Last timestamp processed: {last_ts}")
print("=" * 60)

# 2. EXECUTION

## 2.1 Bronze Reading

In [0]:
bronze_df = spark.read.table(
    bronze_table
)

# ==========================================
# Incremental Processing
# ==========================================

if last_ts:

    bronze_df = bronze_df.filter(

        F.col("ingestion_timestamp")
        >
        F.lit(last_ts)
    )

## 2.2 Data Quality Flags

In [0]:
dq_df = (

    bronze_df

    .withColumn(

        "dq_rule",

        F.when(
            F.col("id").isNull(),
            "null_id"
        )

        .when(
            F.col("name").isNull(),
            "null_name"
        )

        .when(

            (
                F.col("latitude").isNotNull()
            )
            &
            (
                (
                    F.col("latitude") < -90
                )
                |
                (
                    F.col("latitude") > 90
                )
            ),

            "invalid_latitude"
        )

        .when(

            (
                F.col("longitude").isNotNull()
            )
            &
            (
                (
                    F.col("longitude") < -180
                )
                |
                (
                    F.col("longitude") > 180
                )
            ),

            "invalid_longitude"
        )
    )
)

## 2.3 Invalid Records

In [0]:
invalid_df = (

    dq_df

    .filter(
        F.col("dq_rule").isNotNull()
    )

    .withColumn(
        "quarantine_timestamp",
        F.current_timestamp()
    )
)

## 2.4 Valid Records

In [0]:
silver_df = (

    dq_df

    .filter(
        F.col("dq_rule").isNull()
    )

    .drop("dq_rule")
)

## 2.5 Save Quarentine

In [0]:
(
    invalid_df.write

    .format("delta")

    .mode("append")

    .saveAsTable(
        quarantine_table_name
    )
)

## 2.6 Data Quality Metrics

In [0]:
dq_rules = [

    ("null_id",),

    ("null_name",),

    ("invalid_latitude",),

    ("invalid_longitude",)
]

dq_rules_df = spark.createDataFrame(
    dq_rules,
    ["rule_name"]
)

# ==========================================
# Invalids count
# # ==========================================

invalid_metrics_df = (

    invalid_df

    .groupBy("dq_rule")

    .agg(
        F.count("*").alias("invalid_records")
    )

    .withColumnRenamed(
        "dq_rule",
        "rule_name"
    )
)

# ==========================================
# Complies with all rules
# ==========================================

dq_metrics_df = (

    dq_rules_df.alias("rules")

    .join(

        invalid_metrics_df.alias("metrics"),

        on="rule_name",

        how="left"
    )

    .fillna(0, subset=["invalid_records"])

    .withColumn(
        "execution_id",
        F.lit(execution_id)
    )

    .withColumn(
        "pipeline_name",
        F.lit(pipeline_name)
    )

    .withColumn(
        "layer",
        F.lit(layer)
    )

    .withColumn(
        "table_name",
        F.lit("openbrewery_silver")
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )

    .select(

        "execution_id",

        "pipeline_name",

        "layer",

        "table_name",

        "rule_name",

        "invalid_records",

        "created_at"
    )
)

# ==========================================
# Save data quality metrics
# ==========================================

(
    dq_metrics_df.write

    .format("delta")

    .mode("append")

    .saveAsTable(
        data_quality_table_name
    )
)

## 2.7 Standardizations 

In [0]:
silver_df = (

    silver_df

    .withColumn(
        "name",
        F.trim(F.col("name"))
    )

    .withColumn(
        "country",
        F.upper(F.col("country"))
    )

    .withColumn(
        "website_url",
        F.lower(F.col("website_url"))
    )

    .withColumn(

        "phone",

        F.regexp_replace(
            F.col("phone"),
            "[^0-9+]",
            ""
        )
    )
)

## 2.7 Remove duplicates

In [0]:
window_spec = (

    Window

    .partitionBy("id")

    .orderBy(
        F.col("ingestion_timestamp").desc()
    )
)

silver_df = (

    silver_df

    .withColumn(
        "row_num",
        F.row_number().over(window_spec)
    )

    .filter(
        F.col("row_num") == 1
    )

    .drop("row_num")
)

## 2.8 SCD Type 2 Columns

In [0]:
silver_df = (

    silver_df

    .withColumn(
        "valid_from",
        F.current_timestamp()
    )

    .withColumn(
        "valid_to",
        F.lit(None).cast("timestamp")
    )

    .withColumn(
        "is_current",
        F.lit(True)
    )

    .select(

        "id",
        "name",
        "brewery_type",

        "address_1",
        "address_2",
        "address_3",

        "city",
        "state",
        "state_province",
        "postal_code",
        "country",

        "longitude",
        "latitude",

        "phone",
        "website_url",
        "street",

        "record_hash",

        "valid_from",
        "valid_to",
        "is_current",

        "ingestion_timestamp",
        "ingestion_date"
    )
)

## 2.9 Delta Reference

In [0]:
delta_table = DeltaTable.forName(
    spark,
    silver_table
)

current_df = (

    spark.read.table(silver_table)

    .filter(
        F.col("is_current") == True
    )
)

## 2.10 Detect Changes

In [0]:
changes_df = (

    silver_df.alias("source")

    .join(

        current_df.alias("target"),

        on="id",

        how="left"
    )

    .filter(

        (
            F.col("target.id").isNull()
        )

        |

        (
            F.col("source.record_hash")
            !=
            F.col("target.record_hash")
        )
    )

    .select("source.*")
)

## 2.11 Close Old Records

In [0]:
(
    delta_table.alias("target")

    .merge(

        changes_df.alias("source"),

        """
        target.id = source.id
        AND target.is_current = true
        """
    )

    .whenMatchedUpdate(

        condition="""
            target.record_hash <> source.record_hash
        """,

        set={

            "is_current": "false",

            "valid_to": "current_timestamp()"
        }
    )

    .execute()
)

## 2.12 Insert Old Records

In [0]:
(
    changes_df

    .select(
        *spark.read.table(silver_table).columns
    )

    .write

    .mode("append")

    .insertInto(silver_table)
)

## 2.13 Incremental Control

In [0]:
records_processed = changes_df.count()

max_ts = (

    changes_df

    .agg(
        F.max("ingestion_timestamp").alias("max_ts")
    )

    .collect()[0]["max_ts"]
)

## 2.14 Success/Error Monitoring

In [0]:
try:

    end_time = datetime.utcnow()

    monitoring_data = [

        (
            execution_id,
            pipeline_name,
            layer,
            start_time,
            end_time,
            "SUCCESS",
            records_processed,
            None,
            datetime.utcnow()
        )
    ]

    monitoring_df = spark.createDataFrame(
        monitoring_data,
        monitoring_schema
    )

    (
        monitoring_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(
            monitoring_table
        )
    )

except Exception as e:

    end_time = datetime.utcnow()

    monitoring_data = [

        (
            execution_id,
            pipeline_name,
            layer,
            start_time,
            end_time,
            "ERROR",
            0,
            str(e),
            datetime.utcnow()
        )
    ]

    monitoring_df = spark.createDataFrame(
        monitoring_data,
        monitoring_schema
    )

    (
        monitoring_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(
            monitoring_table
        )
    )

    raise e

## 2.15 Save Control

In [0]:
if max_ts:

    control_data = [

        (
            "openbrewery_silver",
            max_ts,
            datetime.utcnow(),
            records_processed
        )
    ]

    control_insert_df = spark.createDataFrame(
        control_data,
        control_schema
    )

    (
        control_insert_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(control_table_name)
    )

## 3.16 Monitoring

In [0]:
try:

    end_time = datetime.utcnow()

    monitoring_data = [

        (
            execution_id,
            pipeline_name,
            layer,
            start_time,
            end_time,
            "SUCCESS",
            records_processed,
            None,
            datetime.utcnow()
        )
    ]

    monitoring_df = spark.createDataFrame(
        monitoring_data,
        monitoring_schema
    )

    (
        monitoring_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(
            monitoring_table
        )
    )

except Exception as e:

    end_time = datetime.utcnow()

    monitoring_data = [

        (
            execution_id,
            pipeline_name,
            layer,
            start_time,
            end_time,
            "ERROR",
            0,
            str(e),
            datetime.utcnow()
        )
    ]

    monitoring_df = spark.createDataFrame(
        monitoring_data,
        monitoring_schema
    )

    (
        monitoring_df.write

        .format("delta")

        .mode("append")

        .saveAsTable(
            monitoring_table
        )
    )

    raise e

## 3. SUMMARY

In [0]:
print("=" * 60)

print("SILVER PROCESSED SUCCESSFULLY")

print(f"Table: {silver_table}")

print(f"Records processed: {records_processed}")

print(f"Last timestamp: {max_ts}")

print("=" * 60)